In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

METRICS_DIR = PROJECT_ROOT / "results" / "metrics"

model_files = {
    "Random Forest": "random_forest_metrics.json",
    "XGBoost": "xgboost_metrics.json",
    "DistilBERT": "distilbert_metrics.json"
}

results = {}

for model_name, filename in model_files.items():
    file_path = METRICS_DIR / filename

    with open(file_path, "r") as f:
        results[model_name] = json.load(f)

print("=" * 55)
print("MODEL METRICS LOADED")
print("=" * 55)

for model_name in results:
    print(model_name, "- Loaded successfully")

print("=" * 55)

MODEL METRICS LOADED
Random Forest - Loaded successfully
XGBoost - Loaded successfully
DistilBERT - Loaded successfully


In [2]:
comparison_rows = []

for model_name, metrics in results.items():
    comparison_rows.append({
        "Model": model_name,
        "Accuracy": metrics.get("accuracy"),
        "Precision": metrics.get("precision"),
        "Recall": metrics.get("recall"),
        "F1-Score": metrics.get("f1"),
        "ROC-AUC": metrics.get("roc_auc"),
        "Training Time (s)": metrics.get("training_time_seconds"),
        "Inference Time (s)": metrics.get("inference_time_seconds")
    })

comparison_df = pd.DataFrame(comparison_rows)

print("=" * 85)
print("FINAL MODEL COMPARISON")
print("=" * 85)

print(comparison_df.to_string(index=False))

print("=" * 85)

FINAL MODEL COMPARISON
        Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC  Training Time (s)  Inference Time (s)
Random Forest  0.934749   0.985812 0.815371       NaN 0.988417                NaN                 NaN
      XGBoost  0.974586   0.979402 0.943360       NaN 0.994276                NaN                 NaN
   DistilBERT  0.970749   0.963485 0.947897  0.955627 0.992477            1074.79             1671.64


In [3]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_path = RESULTS_DIR / "model_comparison.csv"

comparison_df.to_csv(output_path, index=False)

print("=" * 55)
print("MODEL COMPARISON SAVED")
print("=" * 55)
print("File:", output_path)
print("=" * 55)

MODEL COMPARISON SAVED
File: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\model_comparison.csv


In [4]:
for model_name, metrics in results.items():
    print("\n" + "=" * 55)
    print(model_name.upper())
    print("=" * 55)

    for key, value in metrics.items():
        print(f"{key}: {value}")


RANDOM FOREST
model: Random Forest
accuracy: 0.9347489263468758
precision: 0.9858121145624457
recall: 0.8153711353110527
f1_score: 0.8925274461780875
roc_auc: 0.9884166194559163
training_seconds: 89.9
inference_seconds: 1.39
test_samples: 96167

XGBOOST
model: XGBoost
accuracy: 0.974585876652074
precision: 0.9794022092267707
recall: 0.9433596194767806
f1_score: 0.9610431012496812
roc_auc: 0.9942756903019921
training_seconds: 5025.92
inference_seconds: 1.48
test_samples: 96167

DISTILBERT
accuracy: 0.970748801563946
precision: 0.963484843665511
recall: 0.9478971085242208
f1: 0.95562741541131
roc_auc: 0.9924774031257362
inference_time_seconds: 1671.64
test_samples: 96167
training_samples: 10000
training_time_seconds: 1074.79


In [5]:
import pandas as pd
import numpy as np

def get_metric(metrics, *keys):
    for key in keys:
        if key in metrics and metrics[key] is not None:
            return metrics[key]
    return np.nan


comparison_rows = []

for model_name, metrics in results.items():

    precision = get_metric(metrics, "precision", "test_precision")
    recall = get_metric(metrics, "recall", "test_recall")

    f1 = get_metric(
        metrics,
        "f1",
        "f1_score",
        "test_f1",
        "test_f1_score"
    )

    if pd.isna(f1) and pd.notna(precision) and pd.notna(recall):
        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)

    comparison_rows.append({
        "Model": model_name,
        "Accuracy": get_metric(metrics, "accuracy", "test_accuracy"),
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": get_metric(metrics, "roc_auc", "auc", "test_roc_auc"),
        "Training Time (s)": get_metric(
            metrics,
            "training_time_seconds",
            "train_time_seconds",
            "training_time",
            "train_time"
        ),
        "Inference Time (s)": get_metric(
            metrics,
            "inference_time_seconds",
            "prediction_time_seconds",
            "inference_time",
            "test_time_seconds"
        )
    })

comparison_df = pd.DataFrame(comparison_rows)

print("=" * 95)
print("CORRECTED MODEL COMPARISON")
print("=" * 95)

print(comparison_df.round(6).to_string(index=False))

print("=" * 95)

CORRECTED MODEL COMPARISON
        Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC  Training Time (s)  Inference Time (s)
Random Forest  0.934749   0.985812 0.815371  0.892527 0.988417                NaN                 NaN
      XGBoost  0.974586   0.979402 0.943360  0.961043 0.994276                NaN                 NaN
   DistilBERT  0.970749   0.963485 0.947897  0.955627 0.992477            1074.79             1671.64


In [6]:
output_path = PROJECT_ROOT / "results" / "model_comparison.csv"

comparison_df.to_csv(output_path, index=False)

print("Corrected comparison saved:", output_path)

Corrected comparison saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\model_comparison.csv


In [7]:
import pandas as pd
import numpy as np

def get_metric(metrics, *keys):
    for key in keys:
        if key in metrics and metrics[key] is not None:
            return metrics[key]
    return np.nan


comparison_rows = []

for model_name, metrics in results.items():

    comparison_rows.append({
        "Model": model_name,

        "Accuracy": get_metric(
            metrics, "accuracy", "test_accuracy"
        ),

        "Precision": get_metric(
            metrics, "precision", "test_precision"
        ),

        "Recall": get_metric(
            metrics, "recall", "test_recall"
        ),

        "F1-Score": get_metric(
            metrics, "f1", "f1_score", "test_f1"
        ),

        "ROC-AUC": get_metric(
            metrics, "roc_auc", "test_roc_auc"
        ),

        "Training Time (s)": get_metric(
            metrics,
            "training_seconds",
            "training_time_seconds"
        ),

        "Inference Time (s)": get_metric(
            metrics,
            "inference_seconds",
            "inference_time_seconds"
        ),

        "Test Samples": get_metric(
            metrics, "test_samples"
        )
    })

comparison_df = pd.DataFrame(comparison_rows)

print("=" * 110)
print("FINAL MODEL COMPARISON")
print("=" * 110)

print(comparison_df.round(6).to_string(index=False))

print("=" * 110)

FINAL MODEL COMPARISON
        Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC  Training Time (s)  Inference Time (s)  Test Samples
Random Forest  0.934749   0.985812 0.815371  0.892527 0.988417              89.90                1.39         96167
      XGBoost  0.974586   0.979402 0.943360  0.961043 0.994276            5025.92                1.48         96167
   DistilBERT  0.970749   0.963485 0.947897  0.955627 0.992477            1074.79             1671.64         96167


In [8]:
output_path = PROJECT_ROOT / "results" / "model_comparison.csv"

comparison_df.to_csv(output_path, index=False)

print("=" * 55)
print("FINAL MODEL COMPARISON SAVED")
print("=" * 55)
print("File:", output_path)
print("=" * 55)

FINAL MODEL COMPARISON SAVED
File: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\model_comparison.csv
